# Spatial Iteration Engine — Control Panel

Run cells in order. Each panel is a separate cell — **only run the panels you need**.

1. **Setup** — paths and model check
2. **Engine** — create engine and filters
3. **Control Panel** — start/stop, camera, filters, AI (run this one always)
4. **Extra panels** (optional) — Diagnostics, Perception, Filters Designer, Outputs, Performance, Presets

In [1]:
import os, sys

# Repo root (adjust if running from another path)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if not os.path.isdir(os.path.join(REPO_ROOT, "python", "ascii_stream_engine")):
    REPO_ROOT = os.getcwd()
os.chdir(REPO_ROOT)

for p in [os.path.join(REPO_ROOT, "python"), os.path.join(REPO_ROOT, "cpp", "build")]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["ONNX_MODELS_DIR"] = os.path.join(REPO_ROOT, "onnx_models", "mediapipe")
print(f"Repo: {REPO_ROOT}")

# Quick model check (no heavy imports)
for m in ["face_detection_yunet.onnx", "hand_landmarker.task", "pose_landmarker_lite.task"]:
    path = os.path.join(os.environ["ONNX_MODELS_DIR"], m)
    status = f"OK ({os.path.getsize(path)/1024/1024:.1f}MB)" if os.path.exists(path) else "NOT FOUND"
    print(f"  {m}: {status}")

Repo: /home/fissure/repos/Spatial-Iteration-Engine
  face_detection_yunet.onnx: OK (0.2MB)
  hand_landmarker.task: OK (7.5MB)
  pose_landmarker_lite.task: OK (5.5MB)


In [2]:
print("Loading engine...")

from ascii_stream_engine.presentation.notebook_api import (
    build_engine_for_notebook,
    build_general_control_panel,
    build_advanced_diagnostics_panel,
    build_perception_control_panel,
    build_filter_designer_panel,
    build_output_manager_panel,
    build_performance_monitor_panel,
    build_preset_manager_panel,
)

engine = build_engine_for_notebook(camera_index=0)

print(f"Engine created. Mode: {'Graph' if engine._use_graph else 'Legacy'}")
if engine.analyzer_pipeline is not None:
    names = [a.name for a in engine.analyzer_pipeline._analyzers]
    print(f"Perception: {', '.join(names)}")

from ascii_stream_engine.adapters.processors.filters import (
    BloomFilter, BrightnessFilter, ChromaticAberrationFilter,
    EdgeFilter, DetailBoostFilter, InvertFilter, EdgeSmoothFilter,
    KaleidoscopeFilter, KuwaharaFilter, OpticalFlowParticlesFilter,
    PhysarumFilter, RadialCollapseFilter, SlitScanFilter,
    StipplingFilter, ToonShadingFilter, UVDisplacementFilter,
    BoidsFilter, CRTGlitchFilter, GeometricPatternFilter,
    CppPhysarumFilter,
    HandSpatialWarpFilter, HandFrameFilter,
)

filters = {
    "brightness": BrightnessFilter(),
    "edges": EdgeFilter(),
    "detail": DetailBoostFilter(),
    "invert": InvertFilter(),
    "edge_smooth": EdgeSmoothFilter(),
    "optical_flow_particles": OpticalFlowParticlesFilter(),
    "physarum": PhysarumFilter(),
    "radial_collapse": RadialCollapseFilter(),
    "stippling": StipplingFilter(),
    "uv_displacement": UVDisplacementFilter(),
    "boids": BoidsFilter(),
    "crt_glitch": CRTGlitchFilter(),
    "geometric_patterns": GeometricPatternFilter(),
    "cpp_physarum": CppPhysarumFilter(),
    "bloom": BloomFilter(),
    "kaleidoscope": KaleidoscopeFilter(),
    "slit_scan": SlitScanFilter(buffer_size=30),
    "toon_shading": ToonShadingFilter(),
    "kuwahara": KuwaharaFilter(),
    "chromatic_aberration": ChromaticAberrationFilter(),
    "hand_spatial_warp": HandSpatialWarpFilter(),
    "hand_frame": HandFrameFilter(
        effect="ascii",
        ascii_font_size=8,
        ascii_color=(255, 255, 255),
        ascii_bg=(0, 0, 0),
    ),
}
print(f"{len(filters)} filters ready.")

Loading engine...


NDI SDK is not available. Install 'ndi-python' and the NDI SDK to use the NDI output.


HTML(value='<b>Preview (image appears after Start)</b>')

Image(value=b'', format='jpeg')

Engine created. Mode: Graph
Perception: face, hands, pose
22 filters ready.


In [3]:
# Main control — Start/Stop, Camera, Filters, View mode, AI
# This is the only panel you NEED. Run extra panels below if you want them.
control = build_general_control_panel(engine, filters)

HTML(value='<b>Status</b>')

HTML(value='<div style="padding:8px 10px; background:#e7f1ff; border-radius:6px; margin:6px 0; border:1px soli…

HTML(value='<b>Controls</b>')

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1773157069.842273    4643 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773157069.866875    4645 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773157069.900323    4645 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


## Optional Panels
Run any of the cells below **only if you need them**. Each one is independent.

In [ ]:
# Perception — analyzer toggles, confidence, visualization
build_perception_control_panel(engine)

In [ ]:
# Filter Designer — per-filter parameter sliders
build_filter_designer_panel(engine, filters)

In [ ]:
# Diagnostics — profiler, memory, CPU, errors
build_advanced_diagnostics_panel(engine)

In [ ]:
# Performance — FPS, budget, per-node graph timings
build_performance_monitor_panel(engine)

In [ ]:
# Outputs — add/remove sinks (UDP, RTSP, recorder, etc.)
build_output_manager_panel(engine)

In [ ]:
# Presets — save/load/import/export configurations
build_preset_manager_panel(engine)